<a href="https://colab.research.google.com/github/vzavala/StatsBook/blob/master/Python/ch6_gibbs_pca_estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Setup.** Run this cell first: it locates the data files this notebook reads. On Google
Colab it downloads them (the `Data/` and `Support/` folders) from the book's GitHub
repository; on your computer, run the notebook from the folder that contains `Data/`.

In [1]:
import os
import sys
path = ''
if 'google.colab' in sys.modules and not os.path.isdir('Data'):
    if not os.path.isdir('statsbook'):
        os.system('git clone -q --depth 1 --filter=blob:none --sparse '
                  'https://github.com/vzavala/StatsBook.git statsbook')
        os.system('git -C statsbook sparse-checkout set Python/Data Python/Support')
    path = 'statsbook/Python/'

Principal component estimation for the Gibbs reactor (Shiyi (Amy) Qin and Victor Z, UW-Madison, 2024).

| Slides | Python | Meaning |
|---|---|---|
| $\mathbf{X}=[1, X_1,\dots,X_5]$ | `X` | intercept, $P$, $CO$, $H_2$, $CH_3OH$ flows, $T$ ($S=250$) |
| $\mathbf{y}$ | `y` | conversion |
| $\hat{\theta}$, $\mathrm{Cov}(\hat{\theta})$ | `theta_hat`, `Cov_theta` | standard estimate |
| $\mathbf{W}_1$, $\boldsymbol{\Lambda}_1$, $n_1$ | `W1`, `Lam1`, `n1` | retained eigenvectors/eigenvalues |
| $\mathbf{T}_1=\mathbf{X}\mathbf{W}_1$ | `T1` | reduced inputs |
| $\hat{\gamma}$, $\hat{\theta}=\mathbf{W}_1\hat{\gamma}$ | `gamma_hat`, `theta_hat_pca` | PCA estimate |

In [2]:
import numpy as np
import matplotlib.pyplot as plt

data = np.hstack((np.loadtxt(path+'Data/gibbs_covariance.dat'),
                  np.loadtxt(path+'Data/gibbs_lowtemp_class.dat')[:, [0]]))
S = len(data)
np.random.seed(2024)
data[:, 1] += np.random.normal(0, 0.05, S)
y = data[:, 1]
X = np.hstack((np.ones((S, 1)), data[:, [0, 2, 3, 4, 5]]))
print(np.round(X[:6], 2))

[[  1.   166.98  25.56 451.12  74.44 583.15]
 [  1.   170.71  24.75 449.51  75.25 583.15]
 [  1.   141.17  32.22 464.44  67.78 583.15]
 [  1.   146.63  30.64 461.28  69.36 583.15]
 [  1.   164.67  26.08 452.16  73.92 583.15]
 [  1.   157.43  27.79 455.59  72.21 583.15]]


Standard MLE: $\hat{\theta}=(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$. The kernel has three
(numerically) zero eigenvalues, so $\mathrm{Cov}(\hat{\theta})=\sigma^2(\mathbf{X}^T\mathbf{X})^{-1}$ is singular.

In [3]:
theta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
SSE = np.mean((X @ theta_hat - y)**2)
lam, W = np.linalg.eigh(X.T @ X)
order = np.argsort(lam)[::-1]
lam, W = lam[order], W[:, order]
print(f"SSE (mean squared error) = {SSE:.3e}")
print("eigenvalues of X^T X:", np.array2string(lam, precision=2))

SSE (mean squared error) = 2.549e-03
eigenvalues of X^T X: [ 1.45e+08  2.36e+05  1.97e+03  2.44e-11 -3.02e-09 -1.54e-08]


/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/1363051762.py:2: RuntimeWarning: divide by zero encountered in matmul
  SSE = np.mean((X @ theta_hat - y)**2)
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/1363051762.py:2: RuntimeWarning: overflow encountered in matmul
  SSE = np.mean((X @ theta_hat - y)**2)
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/1363051762.py:2: RuntimeWarning: invalid value encountered in matmul
  SSE = np.mean((X @ theta_hat - y)**2)


Keep the $n_1=3$ informative eigenvectors, project $\mathbf{T}_1=\mathbf{X}\mathbf{W}_1$, estimate
$\hat{\gamma}=\arg\min\frac{1}{2}\lVert\mathbf{y}-\mathbf{T}_1\gamma\rVert^2$, and map back $\hat{\theta}=\mathbf{W}_1\hat{\gamma}$ with
$\mathrm{Cov}(\hat{\theta})=\sigma^2\mathbf{W}_1\boldsymbol{\Lambda}_1^{-1}\mathbf{W}_1^T$.

In [4]:
n1 = 3
W1, Lam1 = W[:, :n1], np.diag(lam[:n1])
T1 = X @ W1
gamma_hat = np.linalg.solve(T1.T @ T1, T1.T @ y)
theta_hat_pca = W1 @ gamma_hat
e = X @ theta_hat_pca - y
sigma2 = np.var(e)
Cov_theta_pca = sigma2 * W1 @ np.linalg.inv(Lam1) @ W1.T
print(f"SSE (PCA) = {np.mean(e**2):.3e}")
print("V(theta_hat) with PCA:", np.array2string(np.diag(Cov_theta_pca), precision=2))

fig, axs = plt.subplots(1, 2, figsize=(9, 4))
for ax, th, title in ((axs[0], theta_hat, 'standard'), (axs[1], theta_hat_pca, 'PCA')):
    ax.plot(y, X @ th, 'ko', markerfacecolor='w', markersize=4)
    ax.plot([0, 1], [0, 1], 'k')
    ax.set_title(title)
    ax.set_xlabel('$y$'); ax.set_ylabel(r'$\hat{y}$')
    ax.axis([0.2, 1, 0.2, 1])
    ax.grid(True)
plt.tight_layout()
plt.show()

SSE (PCA) = 2.549e-03
V(theta_hat) with PCA: [7.63e-13 3.32e-07 1.84e-07 2.59e-07 2.67e-07 2.60e-07]


/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/2103151002.py:6: RuntimeWarning: divide by zero encountered in matmul
  e = X @ theta_hat_pca - y
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/2103151002.py:6: RuntimeWarning: overflow encountered in matmul
  e = X @ theta_hat_pca - y
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/2103151002.py:6: RuntimeWarning: invalid value encountered in matmul
  e = X @ theta_hat_pca - y
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/2103151002.py:14: RuntimeWarning: divide by zero encountered in matmul
  ax.plot(y, X @ th, 'ko', markerfacecolor='w', markersize=4)
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/2103151002.py:14: RuntimeWarning: overflow encountered in matmul
  ax.plot(y, X @ th, 'ko', markerfacecolor='w', markersize=4)
/var/folders/nc/c3cjtn9d173b4m3s1nfbpw100000gn/T/ipykernel_40798/2103151002.py:14: RuntimeWarning: invalid value encountered in 